# JS09 — Lab 1: Exact NN vs. ANNOY

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Tujuan:** membandingkan hasil dan waktu pencarian *exact nearest neighbor* (brute force) dengan **ANNOY** pada 1.000 titik acak 2D, dan mengukur seberapa akurat hasil *approximate*-nya.

**Prasyarat:** library `annoy` (dipasang otomatis oleh Langkah 1). Tidak ada dataset eksternal — data dibangkitkan dengan `np.random.seed(42)`.

> Jalankan di **Google Colab**: `Runtime` ▸ `Run all`. Notebook ini cukup memakai CPU, tidak perlu GPU.


# Lab 1 — Exact NN vs. ANNOY

Modul: `JS09 › Lab 1 — Exact NN vs. ANNOY` (polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-mah)

Pada percobaan ini kita membandingkan **exact nearest neighbor** (brute force) dengan **ANNOY**. Data yang dipakai: 1.000 titik acak 2D pada ruang 100×100, query berupa salah satu titik data, 3 tetangga terdekat (`k = 3`), metrik **Euclidean**, dan **3 tree** Annoy.


## Langkah 1 — Install Library ANNOY

`!pip install annoy` cukup dijalankan sekali per sesi Colab. Cell di bawah hanya memasang paket bila `annoy` belum tersedia, sehingga aman dijalankan berulang.

In [ ]:
# Install ANNOY
try:
    import annoy  # noqa: F401
except ImportError:
    !pip install -q annoy

## Langkah 2 — Import Library

In [ ]:
# Import Library
import numpy as np
import matplotlib.pyplot as plt
import time

from annoy import AnnoyIndex

## Langkah 3 — Membuat Dataset Dummy

`np.random.seed(42)` membuat data selalu sama pada setiap run. `query` diambil dari salah satu titik data, sehingga jarak tetangga terdekat pada exact NN pasti `0.0`.

In [ ]:
# Build Random Dataset and Query Point
np.random.seed(42)
n_points = 1000
X = np.random.rand(n_points, 2) * 100  # random value at 100x100 space

# Query point (pick 1 random data point)
query = X[np.random.randint(0, n_points)]

In [ ]:
# [tambahan] Cek cepat dataset
print("Shape X:", X.shape, "| dtype:", X.dtype)
print("Query point:", query)
print("Index asal query:", int(np.argmin(np.linalg.norm(X - query, axis=1))))

## Langkah 4 — Exact NN

Semua jarak dihitung satu per satu (*brute force*) ke seluruh 1.000 titik, lalu diurutkan. Perhatikan waktu komputasinya.

In [ ]:
# Compute Exact NN Using Brute Force
# It will visit the data one by one
start = time.time()
distances = np.linalg.norm(X - query, axis=1)
idx_exact = np.argsort(distances)[:3]  # ambil 3 terdekat
time_exact = time.time() - start

print("Exact NN index:", idx_exact)
print("Exact NN jarak:", distances[idx_exact])
print("Waktu Exact:", round(time_exact*1000, 4), "ms")

**Contoh output modul** (indeks sama karena seed-nya sama; waktu bisa berbeda di mesin Anda):

```text
Exact NN index: [219 898 593]
Exact NN jarak: [0.         1.36915938 2.27931544]
Waktu Exact: 9.1734 ms
```

## Langkah 5 — Perhitungan Jarak dengan ANNOY

ANNOY membangun 3 pohon hyperplane acak, lalu pencarian hanya menyusuri leaf yang relevan. `get_nns_by_vector` secara default memakai `search_k = n_trees × k`, sehingga hasilnya bisa sedikit berbeda dari exact NN — itulah sifat *approximate*-nya.

In [ ]:
# ANNOY 3 Tree
f = 2  # dimensi
t = AnnoyIndex(f, 'euclidean')
for i, vec in enumerate(X):
    t.add_item(i, vec)

t.build(3)  # 3 trees

start = time.time()
idx_ann = t.get_nns_by_vector(query, 3)  # find 3 NN
time_ann = time.time() - start

print("\nAnnoy NN index:", idx_ann)
print("Annoy NN jarak:", [np.linalg.norm(X[i]-query) for i in idx_ann])
print("Waktu Annoy:", round(time_ann*1000, 4), "ms")

**Contoh output modul:**

```text
Annoy NN index: [219, 898, 770]
Annoy NN jarak: [np.float64(0.0), np.float64(1.369159376273702), np.float64(2.568167959732514)]
Waktu Annoy: 0.1373 ms
```

Modul menyoroti selisih waktu yang besar: ANNOY ± **0,13 ms** vs exact NN ± **9,17 ms**. Perlu diingat, waktu exact NN di atas juga mencakup `np.argsort` untuk 1.000 jarak, bukan hanya perhitungan jaraknya — jadi angka itu bukan perbandingan "murni" kecepatan query.

## Langkah 6 — Visualisasi Hasil NN

Bandingkan posisi titik yang dipilih exact NN (biru) dengan perkiraan ANNOY (hijau).

In [ ]:
# Visualize
# Knowing the NN produced by Exact NN and ANNOY
plt.figure(figsize=(6,6))
plt.scatter(X[:,0], X[:,1], c="lightgray", s=20, label="Dataset")
plt.scatter(query[0], query[1], c="red", marker="x", s=100, label="Query")

# Exact NN ditandai biru
plt.scatter(X[idx_exact,0], X[idx_exact,1], c="blue", s=80, label="Exact NN")

# Annoy NN ditandai hijau
plt.scatter(X[idx_ann,0], X[idx_ann,1], c="green", s=50, marker="s", label="Annoy NN")

plt.legend()
plt.title("Exact NN vs Annoy NN (3 trees)")
plt.show()

**Analisis:** TBD

*(Bandingkan `idx_exact` dan `idx_ann`: berapa indeks yang sama? Titik mana yang berbeda, dan berapa selisih jaraknya? Jelaskan mengapa hasil ANNOY masih memadai untuk pencarian pada data besar.)*

In [ ]:
# [tambahan] Berapa banyak tetangga ANNOY yang sama dengan exact NN?
sama = set(int(i) for i in idx_exact) & set(int(i) for i in idx_ann)
print("Irisan indeks:", sorted(sama), f"({len(sama)}/3)")
print("Recall@3:", len(sama) / 3)

### [tambahan] Pengaruh jumlah tree terhadap recall dan waktu

Modul hanya memakai 3 tree. Di sini jumlah tree divariasikan (1, 3, 5, 10, 50) dan diukur **recall@3** terhadap exact NN serta waktu query per titik, memakai 200 query acak.

In [ ]:
# [tambahan] Scan jumlah tree: recall@3 vs waktu query
import pandas as pd

def exact_nn(query_point, k=3):
    d = np.linalg.norm(X - query_point, axis=1)
    return np.argsort(d)[:k]

rng = np.random.default_rng(0)
qidx = rng.choice(n_points, size=200, replace=False)
gt = [exact_nn(X[i]) for i in qidx]

rows = []
for n_trees in [1, 3, 5, 10, 50]:
    t = AnnoyIndex(2, 'euclidean')
    for i, vec in enumerate(X):
        t.add_item(i, vec)
    t.build(n_trees)

    start = time.time()
    preds = [t.get_nns_by_vector(X[i], 3) for i in qidx]
    elapsed = (time.time() - start) * 1000 / len(qidx)

    rec = np.mean([len(set(p) & set(g)) / 3 for p, g in zip(preds, gt)])
    rows.append({'n_trees': n_trees, 'recall@3': rec, 'ms/query': elapsed})

hasil_tree = pd.DataFrame(rows).round(4)
hasil_tree

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(hasil_tree['n_trees'], hasil_tree['recall@3'], marker='o', color='tab:green')
ax1.set_xlabel('Jumlah tree')
ax1.set_ylabel('Recall@3', color='tab:green')
ax1.set_xscale('log')
ax1.set_xticks(hasil_tree['n_trees'])
ax1.set_xticklabels(hasil_tree['n_trees'])

ax2 = ax1.twinx()
ax2.plot(hasil_tree['n_trees'], hasil_tree['ms/query'], marker='s', color='tab:red')
ax2.set_ylabel('Waktu query (ms/query)', color='tab:red')

plt.title('ANNOY: pengaruh jumlah tree (1.000 titik, 2D)')
plt.tight_layout()
plt.show()

## Analisis Lab 1

1. **Irisan hasil ANNOY vs exact NN:** TBD
2. **Selisih jarak pada tetangga yang berbeda:** TBD
3. **Kecenderungan recall saat jumlah tree naik:** TBD
4. **Kapan hasil *approximate* seperti ini masih dapat diterima?** TBD

*(Gunakan tabel dan grafik `[tambahan]` di atas sebagai bukti.)*

---
## Kesimpulan Lab 1

**Perbedaan hasil exact NN vs ANNOY (indeks & jarak):** TBD

**Perbandingan waktu query:** TBD

**Pengaruh jumlah tree terhadap recall:** TBD

**Kesimpulan umum:** TBD